# VI-DOCK API Server

This notebook runs the backend API server of VI-DOCK on Google Colab with GPU acceleration. It includes LightDock for real Protein-Protein Docking, and TxAgent / Hugging Face serverless APIs for the AI Copilot.

### 1. Install System & Molecular Libraries

In [ ]:
!apt-get update -qq
!apt-get install -y -qq openbabel libxrender1 libxext6 libgl1-mesa-glx > /dev/null

### 3. Install Python Dependencies & AI Agent Libraries

In [ ]:
!apt-get install -y -qq openbabel libxrender1 libxext6 libgl1-mesa-glx > /dev/null
!pip install -q fastapi "uvicorn[standard]" python-multipart rdkit meeko requests httpx
!pip install -q tooluniverse txagent openai
!pip install -q lightdock

### 4. Clone Sandbox Branch

In [ ]:
import os
branch_name = "experimental-agent"
repo_url = "https://github.com/messiay/simdock-pro.git"

if not os.path.exists('simdock-pro'):
    print(f"Cloning branch {branch_name}...")
    !git clone -b {branch_name} {repo_url}
else:
    print("Repository already exists. Updating...")
    %cd simdock-pro
    !git checkout {branch_name}
    !git pull
    %cd ..

# Navigate to backend directory
os.chdir('/content/simdock-pro/VI-DOCK/backend')

### 5. Download Docking Binaries

In [ ]:
print("Downloading Vina and Smina...")
!mkdir -p bin
!wget -q https://github.com/ccsb-scripps/AutoDock-Vina/releases/download/v1.2.5/vina_1.2.5_linux_x86_64 -O bin/vina
!wget -q https://github.com/gnina/smina/releases/download/v2020.12.10/smina.static -O bin/smina
!chmod +x bin/vina bin/smina

### 6. Start API Server & Tunnel

In [ ]:
print("Downloading Cloudflare Tunnel...")
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64

import subprocess
import time
import sys

def print_flush(text):
    print(text)
    sys.stdout.flush()

print_flush("\n--- Cleaning up previous runs ---")
os.system("pkill -f uvicorn")
os.system("pkill -f cloudflared")
time.sleep(2)

print_flush("\nStarting VI DOCK API Server on port 8123...")
os.system("nohup uvicorn api.main:app --host 0.0.0.0 --port 8123 > server.log 2>&1 &")

time.sleep(5)

verify_cmd = "curl -m 5 -s http://localhost:8123/ > /dev/null"
if os.system(verify_cmd) != 0:
    print_flush("⚠️ SERVER FAILED TO START! Checking logs...")
    os.system("cat server.log")
else:
    print_flush("✅ Server is running locally on port 8123")

print_flush("\n--- DEPLOYMENT COMPLETE ---")
print_flush("Starting Cloudflare secure tunnel...")
os.system("nohup ./cloudflared-linux-amd64 tunnel --url http://127.0.0.1:8123 > cloudflare.log 2>&1 &")
time.sleep(5)

# Parse the log to find the URL
try:
    with open("cloudflare.log", "r") as f:
        log_content = f.read()
        import re
        url_match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', log_content)
        if url_match:
            print_flush("\n" + "="*50)
            print_flush(f"✅ YOUR SANDBOX API URL IS:\n{url_match.group(0)}")
            print_flush("="*50)
        else:
            print_flush("\n⚠️ Still waiting for URL... Here are recent logs:")
            os.system("cat cloudflare.log | tail -n 10")
except Exception as e:
    print_flush(f"Error reading Cloudflare log: {e}")

# Keep the cell running
while True:
    time.sleep(60)